# ThreadAtlas on OmniMemEval

OmniMemEval's own harness, at a pinned commit, run on ThreadAtlas through its `threadatlas` adapter:
LongMemEval (cleaned, 500 questions) and LoCoMo (10 conversations, category 5 left out, as the harness
scores it). The answer model is `gpt-4.1-mini-2025-04-14` and the judge `gpt-4o-mini-2024-07-18`, the
harness's own settings, with `TOPK=20`.

```
the harness at its commit, with the adapter's files laid over it
the store, Step 0's export and longmemeval_s.json from the attached inputs
the datasets by the harness's own prepare scripts
for each benchmark: the harness's run script, all six steps
```

`RUN = "smoke"` runs 20 LongMemEval users and the first LoCoMo conversation; `"full"` runs every one.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

RUN = "full"
BENCHMARKS = ["locomo"]
LABEL = "ta021-41mini"            # the retrieval the store gives: ta020 without chunks, ta021 with
READ_MODEL = "gpt-4.1-mini-2025-04-14"   # the model that reads new sessions; the package default is gpt-5.6-luna
COMMIT = "7c79d46e31dd85f3da2509b430bba9406656dd28"
WORK = Path("/kaggle/working")
HARNESS = WORK / "omnimemeval"


def found(name):
    """The first file of this name under the attached inputs."""
    for path in sorted(Path("/kaggle/input").rglob(name)):
        return path
    raise SystemExit(name + " is not attached")


# the kit: our package and the adapter's files, unzipped where Kaggle left them zipped
KIT = WORK / "kit"
KIT.mkdir(exist_ok=True)
for archive in Path("/kaggle/input").rglob("*.zip"):
    if archive.stem in ("threadatlas_memory", "adapter"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(KIT / archive.stem)
for folder in Path("/kaggle/input").rglob("threadatlas_memory"):
    if folder.is_dir() and (folder / "memory.py").exists() and not (KIT / "threadatlas_memory" / "memory.py").exists():
        shutil.copytree(folder, KIT / "threadatlas_memory", dirs_exist_ok=True)
for folder in Path("/kaggle/input").rglob("adapter"):
    if folder.is_dir() and not (KIT / "adapter" / "scripts").exists():
        shutil.copytree(folder, KIT / "adapter", dirs_exist_ok=True)
if (KIT / "threadatlas_memory" / "threadatlas_memory").exists():      # a zip that holds its own folder
    shutil.move(str(KIT / "threadatlas_memory"), str(KIT / "package"))
    shutil.move(str(KIT / "package" / "threadatlas_memory"), str(KIT / "threadatlas_memory"))
if (KIT / "adapter" / "adapter").exists():
    shutil.move(str(KIT / "adapter"), str(KIT / "outer"))
    shutil.move(str(KIT / "outer" / "adapter"), str(KIT / "adapter"))

STORE = found("threadatlas.sqlite").parent
STEP0 = found("units.jsonl").parent
SOURCE = found("longmemeval_s.json")
print("store:", STORE, "| Step 0:", STEP0, "| source:", SOURCE)

# the key, from the Kaggle secret, for the reading model, the answer model and the judge
key = os.environ.get("OPENAI_API_KEY")
if not key:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("OPENAI_API_KEY")
for name in ("OPENAI_API_KEY", "ANSWER_API_KEY", "EVAL_API_KEY"):
    os.environ[name] = key

In [ ]:
# the harness at its commit, the adapter laid over it, and what it needs
if not HARNESS.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/MemTensor/OmniMemEval.git", str(HARNESS)], check=True)
subprocess.run(["git", "-C", str(HARNESS), "checkout", "-q", COMMIT], check=True)
for path in (KIT / "adapter").rglob("*"):
    if path.is_file():
        target = HARNESS / path.relative_to(KIT / "adapter")
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, target)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(HARNESS / "requirements_user_memory.txt"),
                "fastembed", "tiktoken"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "scripts/tests/unit/test_threadatlas_client.py"], cwd=HARNESS, check=True)

subprocess.run([sys.executable, "data/longmemeval/prepare_longmemeval.py"], cwd=HARNESS, check=True)
subprocess.run([sys.executable, "data/locomo/prepare_locomo.py"], cwd=HARNESS, check=True)

if RUN == "smoke":                # the first users only, so a whole pass of every step is minutes
    lme = HARNESS / "data" / "longmemeval" / "longmemeval_s_cleaned.json"
    lme.write_text(json.dumps(json.loads(lme.read_text(encoding="utf-8"))[:20]), encoding="utf-8")
    locomo = HARNESS / "data" / "locomo" / "locomo10.json"
    locomo.write_text(json.dumps(json.loads(locomo.read_text(encoding="utf-8"))[:1]), encoding="utf-8")

ENV = HARNESS / ".env.threadatlas"
ENV.write_text("\n".join([
    'THREADATLAS_REPO="' + str(KIT) + '"',
    'THREADATLAS_STATE_DIR="' + str(WORK / "state") + '"',
    'THREADATLAS_STORE_DIR="' + str(STORE) + '"',
    'THREADATLAS_STEP0_DIR="' + str(STEP0) + '"',
    'THREADATLAS_SOURCE="' + str(SOURCE) + '"',
    'THREADATLAS_READ_MODEL="' + READ_MODEL + '"',
    "LLM_WORKERS=10",
    "TOPK=20",
    'ANSWER_MODEL="gpt-4.1-mini-2025-04-14"',
    'ANSWER_BASE_URL="https://api.openai.com/v1"',
    'EVAL_MODEL="gpt-4o-mini-2024-07-18"',
    'EVAL_BASE_URL="https://api.openai.com/v1"',
]) + "\n", encoding="utf-8")

In [ ]:
# each benchmark through the harness's own run script, all six steps
WORKERS = {"lme": "4", "locomo": "10"}
SCRIPT = {"lme": "scripts/run_lme_eval.sh", "locomo": "scripts/run_locomo_eval.sh"}
for benchmark in BENCHMARKS:
    version = LABEL + "-" + RUN
    command = ["bash", SCRIPT[benchmark], "--lib", "threadatlas", "--env", str(ENV), "--version", version,
               "--workers", WORKERS[benchmark]]
    print(" ".join(command))
    with open(WORK / (benchmark + "-" + version + ".log"), "w", encoding="utf-8") as log:
        done = subprocess.run(command, cwd=HARNESS, stdout=log, stderr=subprocess.STDOUT)
    print(benchmark, "exit", done.returncode)
    lines = (WORK / (benchmark + "-" + version + ".log")).read_text(encoding="utf-8", errors="replace").splitlines()
    for line in lines[-25:]:
        print(line)

In [ ]:
# the harness's results, beside this notebook's output
if (HARNESS / "results").exists():
    shutil.copytree(HARNESS / "results", WORK / "results", dirs_exist_ok=True)
for path in sorted((WORK / "results").rglob("*metric*.json")):
    print(path, path.read_text(encoding="utf-8")[:1500])